## tl;dr
本文件顺序复查字面示例、三个真实小窗口和图像测量边界。只读，不生成结构信号。

## Context & Methods
无过滤的局部转折；后续已完成点确认此前峰。图像像素不是原始指标。
### Key Assumptions
原三个JSON及图片SHA不变；时间为上海交易时间。

In [ ]:
from pathlib import Path
import json, sys
root = Path('/Users/congming/github/goldenshare')
sys.path.insert(0, str(root))
from src.biz.services.wealth.market.index_detail_daily_wave.local_extrema import scan_values
from scripts.research.index_market.xxm.local_extrema_study import CASES, read_case, summarize_case
example = scan_values([2,5,8,6,3,4,6,4])
assert [(p.kind, str(p.value)) for p in example] == [('MAX','8'),('MIN','3'),('MAX','6')]
print([(p.kind, str(p.value), p.first_index, p.confirmed_index) for p in example])

## Data
三个既存窗口共198行；各自路径与SHA在CASES中，旧/新路径在下面的比较单元逐项对应；未读取正式源。

In [ ]:
recomputed = []
for case in CASES:
    rows, quality = read_case(case)
    result = summarize_case(rows, case, quality)
    recomputed.append(result)
    print(case.name, quality['rows'], result['count'], result['independentPrefixChecks'])

## Results
以下输出与完成的试验文件比对；不会将单元测试通过解释为作者一致性通过。

In [ ]:
report = Path('/Users/congming/github/goldenshare/reports/xxm/local-extrema')
source_paths = {'m120_2023': ('reports/xxm_m120_reference_trace_20261020_01/source_window.json',
               'reports/xxm/sources/xxm_m120_reference_trace_20261020_01/source_window.json'),
 'm90_2023': ('reports/xxm_same_day_m90_m120_20230301_20261020_01/m90_source_window.json',
              'reports/xxm/sources/xxm_same_day_m90_m120_20230301_20261020_01/m90_source_window.json'),
 'm90_2026': ('reports/xxm_feb25_m90_reference_evidence_20261021_01/source_projection.json',
              'reports/xxm/sources/xxm_feb25_m90_reference_evidence_20261021_01/source_projection.json')}
for result in recomputed:
    saved = json.loads((report / (result['case'] + '.json')).read_text())
    old_path, new_path = source_paths[result['case']]
    assert saved['source'] == old_path
    assert result['source'] == new_path
    saved_values, current_values = dict(saved), dict(result)
    del saved_values['source']
    del current_values['source']
    assert saved_values == current_values
image = json.loads((report / 'image_measurement.json').read_text())
print('图片独立三峰数值验证:', image['independentThreePeakValidation'])
for variant in image['variants']:
    print(variant['spec']['name'], '柱组', len(variant['bars']), '遮挡', variant['censored_bars'], '断点', len(variant['gaps']))

## Takeaways
基础数学行为与三个小窗口可复查；额外转折全部保留。盘口原图遮挡及未知原始柱值限制作者一致性判断；下一步不能只凭程序运行成功就迁移产品。